# ContextEngine — Basic and Advanced RAG

This notebook builds a retrieval pipeline in two stages:

1. **Basic RAG** — dense semantic retrieval using sentence embeddings and cosine similarity.
2. **Advanced RAG** — hybrid retrieval using BM25, Reciprocal Rank Fusion, and cross-encoder reranking.

The focus is the retrieval stack: how candidate selection changes before grounded LLM generation.

## 1. Setup

Install the runtime dependencies and import the embedding model, BM25 retriever, cross-encoder reranker, NumPy, and Gemini SDK.

In [ ]:
!pip install -q google-genai sentence-transformers rank_bm25

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from google.colab import files, userdata
from google import genai
import numpy as np

## 2. Upload and read a document

Colab returns uploaded files as bytes, so decode the selected file into UTF-8 text.

In [ ]:
uploaded = files.upload()

filename = list(uploaded.keys())[0]
text = uploaded[filename].decode("utf-8")

print("Uploaded file:", filename)
print(text)

## 3. Chunk the document

Split the document into overlapping word-based chunks. Smaller chunks improve retrieval precision, while overlap helps preserve context across boundaries.

In [ ]:
words = text.split()

chunks = []
chunk_size = 40
overlap = 10
start = 0

while start < len(words):
    chunk_words = words[start:start + chunk_size]
    chunk_text = " ".join(chunk_words)
    chunks.append(chunk_text)
    start = start + (chunk_size - overlap)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}:")
    print(chunk)

# Part A — Basic RAG

Basic RAG uses one retrieval signal: dense semantic similarity.

## 4. Embed document chunks

Each chunk is independently encoded into the same dense vector space.

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedding_model.encode(chunks)

print("Embedding shape:", chunk_embeddings.shape)

## 5. Embed the user query

In [ ]:
query = input("Enter your query: ")
query_embedding = embedding_model.encode(query)

print("Query embedding shape:", query_embedding.shape)

## 6. Semantic search with cosine similarity

Score the query against every chunk and rank the chunks from most to least semantically similar.

In [ ]:
semantic_results = []

for i in range(len(chunk_embeddings)):
    dot = np.dot(query_embedding, chunk_embeddings[i])
    query_magnitude = np.linalg.norm(query_embedding)
    chunk_magnitude = np.linalg.norm(chunk_embeddings[i])

    cosine_similarity = dot / (query_magnitude * chunk_magnitude)
    semantic_results.append((i, cosine_similarity))

semantic_ranked = sorted(
    semantic_results,
    key=lambda x: x[1],
    reverse=True
)

print("Semantic ranking:")
print(semantic_ranked)

## 7. Build Basic RAG context

Take the strongest semantic chunks and join them into the context passed to the LLM.

In [ ]:
basic_k = 3
basic_top_results = semantic_ranked[:basic_k]

basic_chunks = []

for chunk_index, similarity in basic_top_results:
    basic_chunks.append(chunks[chunk_index])

basic_retrieved_context = "\n\n".join(basic_chunks)

print(basic_retrieved_context)

## 8. Ground the generation prompt

In [ ]:
basic_prompt = f"""
Answer the question using ONLY the context provided below.

If the answer is not present in the context, say:
"I don't have enough information to answer that."

Context:
{basic_retrieved_context}

Question:
{query}

Answer:
"""

## 9. Generate the Basic RAG answer

Store `GEMINI_API_KEY` in Colab Secrets rather than hardcoding it into the notebook.

In [ ]:
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

basic_interaction = client.interactions.create(
    model="gemini-3.8-flash",
    input=basic_prompt
)

basic_answer = basic_interaction.output_text

print("Basic RAG answer:")
print(basic_answer)

# Part B — Advanced RAG

The advanced pipeline adds independent lexical retrieval, rank fusion, and pairwise reranking.

```text
Dense retrieval + BM25 → RRF → Cross-Encoder → Final top-k → Context → LLM
```

## 10. Build the BM25 lexical index

BM25 operates over tokenized text rather than dense vectors.

In [ ]:
tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created for", len(tokenized_chunks), "chunks")

## 11. Run BM25 retrieval

Score the same query against every chunk using the lexical index.

In [ ]:
tokenized_query = query.lower().split()
bm25_scores = bm25.get_scores(tokenized_query)

bm25_results = []

for index, score in enumerate(bm25_scores):
    bm25_results.append((index, score))

bm25_ranked = sorted(
    bm25_results,
    key=lambda x: x[1],
    reverse=True
)

print("BM25 ranking:")
print(bm25_ranked)

## 12. Reciprocal Rank Fusion

Dense similarity and BM25 scores live on different scales. RRF avoids direct score normalization by combining **rank positions** instead.

```text
RRF contribution = 1 / (60 + rank)
```

In [ ]:
rrf_scores = {}

for rank, item in enumerate(semantic_ranked, start=1):
    chunk_index = item[0]
    rrf_scores[chunk_index] = (
        rrf_scores.get(chunk_index, 0)
        + 1 / (60 + rank)
    )

for rank, item in enumerate(bm25_ranked, start=1):
    chunk_index = item[0]
    rrf_scores[chunk_index] = (
        rrf_scores.get(chunk_index, 0)
        + 1 / (60 + rank)
    )

fused_results = sorted(
    rrf_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print("RRF fused ranking:")
print(fused_results)

## 13. Build query–candidate pairs

RRF outputs chunk indices and fusion scores. Recover the original chunk text so the cross-encoder can read the query and candidate together.

In [ ]:
pairs = []

for chunk_index, rrf_score in fused_results:
    pairs.append((query, chunks[chunk_index]))

print("Number of query-chunk pairs:", len(pairs))

## 14. Cross-encoder reranking

A cross-encoder performs pairwise query-document relevance scoring. Unlike bi-encoder retrieval, the query and candidate are processed jointly.

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
cross_encoder_scores = reranker.predict(pairs)

print("Cross-encoder scores:")
print(cross_encoder_scores)

## 15. Map reranker scores back to chunk indices

The output scores follow the same order as the input pairs. Reattach each score to its original candidate and sort again.

In [ ]:
reranked_results = []

for i in range(len(fused_results)):
    chunk_index = fused_results[i][0]
    score = cross_encoder_scores[i]
    reranked_results.append((chunk_index, score))

reranked_results = sorted(
    reranked_results,
    key=lambda x: x[1],
    reverse=True
)

print("Final reranked results:")
print(reranked_results)

## 16. Build the final Advanced RAG context

In [ ]:
advanced_k = 3
top_reranked = reranked_results[:advanced_k]

advanced_chunks = []

for chunk_index, score in top_reranked:
    advanced_chunks.append(chunks[chunk_index])

advanced_retrieved_context = "\n\n".join(advanced_chunks)

print(advanced_retrieved_context)

## 17. Ground and generate the Advanced RAG answer

In [ ]:
advanced_prompt = f"""
Answer the question using ONLY the context provided below.

If the answer is not present in the context, say:
"I don't have enough information to answer that."

Context:
{advanced_retrieved_context}

Question:
{query}

Answer:
"""

advanced_interaction = client.interactions.create(
    model="gemini-3.8-flash",
    input=advanced_prompt
)

advanced_answer = advanced_interaction.output_text

print("Advanced RAG answer:")
print(advanced_answer)

## 18. Simple evaluation

Inspect three distinct surfaces:

- **Retrieval relevance** — Did the selected chunks contain the necessary evidence?
- **Faithfulness** — Is the generated answer supported by the retrieved context?
- **Answer correctness** — Does the final answer match the expected answer?

In [ ]:
print("=== BASIC RAG ===")
print("Retrieved context:")
print(basic_retrieved_context)
print("\nAnswer:")
print(basic_answer)

print("\n=== ADVANCED RAG ===")
print("Retrieved context:")
print(advanced_retrieved_context)
print("\nAnswer:")
print(advanced_answer)

# Pipeline Flow Diagrams

## Basic RAG

```text
Document
   ↓
Chunk
   ↓
Embed chunks
   ↓
User query → Embed query
   ↓
Cosine similarity
   ↓
Semantic ranking
   ↓
Top-k chunks
   ↓
Context
   ↓
Grounded prompt
   ↓
LLM
   ↓
Answer
```

## Advanced RAG

```text
                         ┌─────────────────────┐
                         │      User Query     │
                         └──────────┬──────────┘
                                    │
                     ┌──────────────┴──────────────┐
                     │                             │
                     ↓                             ↓
              Dense Semantic Search          BM25 Lexical Search
                     │                             │
                     ↓                             ↓
              Semantic Ranking                BM25 Ranking
                     └──────────────┬──────────────┘
                                    ↓
                          Reciprocal Rank Fusion
                                    ↓
                           Candidate Ranking
                                    ↓
                            Query + Chunk Pairs
                                    ↓
                         Cross-Encoder Reranker
                                    ↓
                           Final Relevance Rank
                                    ↓
                              Final Top-k
                                    ↓
                                Context
                                    ↓
                           Grounded Generation
                                    ↓
                                 Answer
```

### Mental model

```text
                     ┌→ DENSE SEARCH ─┐
QUERY ───────────────┤                 ├→ RRF → RERANKER → TOP-K → CONTEXT → LLM
                     └→ BM25 SEARCH ──┘
```

The generation stage stays almost the same. The main engineering improvement is the retrieval stack that decides **which evidence reaches the model**.